# Quantile model for shape metrics

In [ ]:
import sys
import os
import warnings

import numpy as np
import torch
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

warnings.filterwarnings("ignore")

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load data and model

In [ ]:
X = pd.read_csv("../_temp/X.csv")
y = pd.read_csv("../_temp/y.csv")
Xpred = pd.read_csv("../_temp/Xpred_1D.csv", index_col=[0, 1, 2, 3])
H_mean = np.load("../_temp/H.prior_mean.Xpred_1D.npy")
phi_mean = np.load("../_temp/phi.prior_mean.Xpred_1D.npy")
H_quantiles = np.load("../_temp/H.quantiles.Xpred_1D.npy")
phi_quantiles = np.load("../_temp/phi.quantiles.Xpred_1D.npy")

## Plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

In [ ]:
Rgt_pred = Xpred["Gap_to_thickness_ratio"].unique()
Cas = Xpred["Capillary_number"].unique()
Slurries = Xpred.index.get_level_values("Slurry").unique()

Cas_sorted = np.sort(Cas)
cmap = plt.get_cmap("viridis", len(Cas_sorted))
norm = mcolors.BoundaryNorm(
    np.concatenate(
        [
            [Cas_sorted[0] * 0.9],
            (Cas_sorted[:-1] + Cas_sorted[1:]) / 2,
            [Cas_sorted[-1] * 1.1],
        ]
    ),
    ncolors=len(Cas_sorted),
)

## Mean

In [ ]:
fig, axes = plt.subplots(1, len(Slurries), sharex=True, sharey="row")

for slurry, ax in zip(Slurries, axes):
    ok = X["Slurry"] == slurry
    this_X = X[ok]
    this_y = y[ok]

    ok_pred = Xpred.index.get_level_values("Slurry") == slurry
    this_Xpred = Xpred[ok_pred]
    this_mean = H_mean[ok_pred]

    for ca in this_X["Capillary_number"].unique():
        ok = this_X["Capillary_number"] == ca
        ok_pred = this_Xpred["Capillary_number"] == ca

        ax.scatter(
            this_X[ok]["Gap_to_thickness_ratio"],
            this_y[ok]["H"],
            color=cmap(norm(ca)),
        )

        ax.plot(
            this_Xpred[ok_pred]["Gap_to_thickness_ratio"],
            this_mean[ok_pred],
            color=cmap(norm(ca)),
        )

    (cos_theta,) = this_X["Cos_theta"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("H")
fig.suptitle("Prior mean")

In [ ]:
fig, axes = plt.subplots(1, len(Slurries), sharex=True, sharey="row")

for slurry, ax in zip(Slurries, axes):
    ok = X["Slurry"] == slurry
    this_X = X[ok]
    this_y = y[ok]

    ok_pred = Xpred.index.get_level_values("Slurry") == slurry
    this_Xpred = Xpred[ok_pred]
    this_mean = phi_mean[ok_pred]

    for ca in this_X["Capillary_number"].unique():
        ok = this_X["Capillary_number"] == ca
        ok_pred = this_Xpred["Capillary_number"] == ca

        ax.scatter(
            this_X[ok]["Gap_to_thickness_ratio"],
            this_y[ok]["phi"],
            color=cmap(norm(ca)),
        )

        ax.plot(
            this_Xpred[ok_pred]["Gap_to_thickness_ratio"],
            this_mean[ok_pred],
            color=cmap(norm(ca)),
        )

    (cos_theta,) = this_X["Cos_theta"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("phi")
fig.suptitle("Prior mean")

## Quantiles

In [ ]:
fig, axes = plt.subplots(1, len(Slurries), sharex=True, sharey="row")

for slurry, ax in zip(Slurries, axes):
    ok = X["Slurry"] == slurry
    this_X = X[ok]
    this_y = y[ok]

    ok_pred = Xpred.index.get_level_values("Slurry") == slurry
    this_Xpred = Xpred[ok_pred]
    this_quantiles = H_quantiles[ok_pred]

    for ca in this_X["Capillary_number"].unique():
        ok = this_X["Capillary_number"] == ca
        ok_pred = this_Xpred["Capillary_number"] == ca

        ax.scatter(
            this_X[ok]["Gap_to_thickness_ratio"],
            this_y[ok]["H"],
            color=cmap(norm(ca)),
        )

        ax.fill_between(
            this_Xpred[ok_pred]["Gap_to_thickness_ratio"],
            this_quantiles[ok_pred, 0],
            this_quantiles[ok_pred, -1],
            facecolor=cmap(norm(ca)),
            edgecolor="none",
            alpha=0.3,
        )

    (cos_theta,) = this_X["Cos_theta"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("H")
fig.suptitle("Quantiles")

In [ ]:
fig, axes = plt.subplots(1, len(Slurries), sharex=True, sharey="row")

for slurry, ax in zip(Slurries, axes):
    ok = X["Slurry"] == slurry
    this_X = X[ok]
    this_y = y[ok]

    ok_pred = Xpred.index.get_level_values("Slurry") == slurry
    this_Xpred = Xpred[ok_pred]
    this_quantiles = phi_quantiles[ok_pred]

    for ca in this_X["Capillary_number"].unique():
        ok = this_X["Capillary_number"] == ca
        ok_pred = this_Xpred["Capillary_number"] == ca

        ax.scatter(
            this_X[ok]["Gap_to_thickness_ratio"],
            this_y[ok]["phi"],
            color=cmap(norm(ca)),
        )

        ax.fill_between(
            this_Xpred[ok_pred]["Gap_to_thickness_ratio"],
            this_quantiles[ok_pred, 0],
            this_quantiles[ok_pred, -1],
            facecolor=cmap(norm(ca)),
            edgecolor="none",
            alpha=0.3,
        )

    (cos_theta,) = this_X["Cos_theta"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("phi")
fig.suptitle("Quantiles")